# Eksplorasi Data Pengangguran Kabupaten Tuban

Notebook awal untuk laporan kerja praktik. Sumber utama: `Datasets/Data_Pengangguran_Clean_Audited_8_Sep_2026.csv`. Analisis ini merangkum jumlah pada data sumber; bukan analisis sebab-akibat.

## Ruang lingkup

Unit baris adalah wilayah desa/kelurahan pada CSV. Notebook memeriksa kelengkapan, menjumlahkan `Total_Penganggur` dan `Total_Penduduk_Bekerja` menurut kecamatan, lalu menampilkan kolom QC. Pastikan periode data, definisi indikator, dan denominator dikonfirmasi sebelum menghitung tingkat atau persentase pengangguran.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from laporan_kerja_praktik import load_dataset, summarize_by_district

data = load_dataset()
print(f"Cakupan: {len(data):,} baris × {len(data.columns)} kolom")
print(f"Jumlah kecamatan: {data['kecamatan'].nunique()}")
print(f"Jenis wilayah: {', '.join(sorted(data['jenis_kd'].dropna().unique()))}")

## Contoh baris

Tampilkan kolom inti saja agar identitas wilayah, jumlah penganggur, dan jumlah penduduk bekerja mudah diperiksa.

In [ ]:
display(
    data[["kecamatan", "kel_desa", "Total_Penganggur", "Total_Penduduk_Bekerja"]]
    .head()
)

## Pemeriksaan nilai kosong

Header `#N/A` berasal dari CSV sumber; jangan menafsirkannya sebagai indikator tanpa klarifikasi.

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
display(missing[missing.gt(0)].to_frame("jumlah_kosong").head(20))

## Ringkasan per kecamatan

Angka di bawah adalah jumlah agregat dari kolom sumber, bukan tingkat pengangguran. Perbandingan jumlah juga dipengaruhi ukuran populasi wilayah.

In [ ]:
ringkasan_kecamatan = summarize_by_district(data)
display(ringkasan_kecamatan)

In [ ]:
top_10 = ringkasan_kecamatan.nlargest(10, "Total_Penganggur").sort_values("Total_Penganggur")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top_10, x="Total_Penganggur", y="kecamatan", color="#4472C4", ax=ax)
ax.set(title="10 kecamatan dengan jumlah penganggur terbesar", xlabel="Jumlah penganggur (sumber)", ylabel="Kecamatan")
fig.tight_layout()
plt.show()

## Ringkasan menurut jenis wilayah

Jumlah baris ditampilkan sebagai hitungan unit yang tercatat. Periksa kamus data dan aturan pencatatan sebelum menyamakan unit antarjenis wilayah.

In [ ]:
ringkasan_jenis = (
    data.groupby("jenis_kd", dropna=False)
    .agg(
        jumlah_baris=("kel_desa", "count"),
        jumlah_penganggur=("Total_Penganggur", "sum"),
        jumlah_bekerja=("Total_Penduduk_Bekerja", "sum"),
    )
    .reset_index()
)
display(ringkasan_jenis)

## Pemeriksaan kualitas sumber

Ringkasan ini mempertahankan nilai QC apa adanya. Tinjau definisi setiap pemeriksaan sebelum memutuskan baris mana yang layak dimasukkan ke analisis.

In [ ]:
kolom_qc = [column for column in data.columns if column.startswith("QC_")]
for column in kolom_qc:
    hitung = data[column].value_counts(dropna=False).rename("jumlah").rename_axis("nilai").reset_index()
    hitung.insert(0, "kolom_qc", column)
    display(hitung)

## Catatan untuk pembahasan laporan

Gunakan tabel dan grafik di atas untuk menyusun pembahasan sesuai rumusan masalah. Cantumkan kecamatan/kelompok yang dibandingkan, satuan hitungan, periode sumber yang telah dikonfirmasi, temuan QC yang relevan, dan batas interpretasi. Untuk proporsi atau tingkat pengangguran, definisikan denominator terlebih dahulu dan jelaskan dasar pemilihannya.